# Basic edge detection across single-dart images

This notebook applies a simple Canny edge detector to every supported image in `data/images/single_dart`. Each result is displayed beside its resized source image. Adjust the Canny thresholds in the setup cell to compare edge sensitivity.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
repo_root = next(
    (path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not find the dartcv project root")

image_dir = repo_root / "data" / "images" / "single_dart"
image_dir = repo_root / "data" / "images" / "multi_dart"
supported_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
image_paths = sorted(
    path
    for path in image_dir.iterdir()
    if path.is_file() and path.suffix.lower() in supported_extensions
)

LOW_THRESHOLD = 50
HIGH_THRESHOLD = 150
MAX_DISPLAY_SIDE = 1000

if not image_paths:
    raise FileNotFoundError(f"No supported images found in {image_dir}")
print(f"Found {len(image_paths)} images in {image_dir.relative_to(repo_root)}")

## Process every image

Canny uses the lower and upper thresholds above after grayscale conversion and Gaussian blur. Lower thresholds tend to retain more edges and noise; higher thresholds retain stronger edges.

In [ ]:
if LOW_THRESHOLD < 0 or HIGH_THRESHOLD <= LOW_THRESHOLD:
    raise ValueError("Canny thresholds must satisfy 0 <= low < high")
if MAX_DISPLAY_SIDE <= 0:
    raise ValueError("MAX_DISPLAY_SIDE must be positive")

processed_count = 0
for image_path in image_paths:
    source = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if source is None:
        print(f"Skipping unreadable image: {image_path.name}")
        continue

    scale = min(1.0, MAX_DISPLAY_SIDE / max(source.shape[:2]))
    if scale < 1.0:
        source = cv2.resize(
            source,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_AREA,
        )

    gray = cv2.cvtColor(source, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, LOW_THRESHOLD, HIGH_THRESHOLD)

    figure, axes = plt.subplots(1, 2, figsize=(14, 6))
    figure.suptitle(image_path.name)
    axes[0].imshow(cv2.cvtColor(source, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Input")
    axes[1].imshow(edges, cmap="gray")
    axes[1].set_title("Canny edges")
    for axis in axes:
        axis.axis("off")
    figure.tight_layout()
    plt.show()
    plt.close(figure)
    processed_count += 1

print(f"Processed {processed_count} of {len(image_paths)} images")